# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}\n")
print(f"Published: {metadata.datePublished}, Version: {metadata.version}\n")
print(f"License: {metadata.license if hasattr(metadata, 'license') else 'N/A'}")
print("Citation:", getattr(metadata, 'citeAs', 'N/A'))

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List available record sets by their @id
record_sets = dataset.record_sets
if not record_sets:
    print("No record sets found in the dataset metadata.")
else:
    print(f"Found {len(record_sets)} record sets:")
    for rs in record_sets:
        print(f"  - RecordSet @id: {rs['@id']} | name: {rs.get('name', '(no name)')}")

# As an example, display the fields of the first record set (if available)
if record_sets:
    # We'll select the first record set for the sake of exploration
    first_rs_id = record_sets[0]['@id']
    # Get record set details
    rs_fields = record_sets[0].get('field', [])
    if rs_fields:
        print(f"\nFields in record set {first_rs_id}:")
        for field in rs_fields:
            # field can be dict or @id string
            if isinstance(field, dict):
                print(f"  - Field @id: {field.get('@id')} | name: {field.get('name', '(no name)')}")
            else:
                print(f"  - Field @id: {field}")
    else:
        print(f"No fields found for record set {first_rs_id}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# If any record sets are found, load their data
dfs = {}
if record_sets:
    record_set_ids = [rs['@id'] for rs in record_sets]
    print(f"\nLoading data for record sets: {record_set_ids}\n")
    for record_set_id in record_set_ids:
        try:
            records = list(dataset.records(record_set=record_set_id))
            if records:
                df = pd.DataFrame(records)
                dfs[record_set_id] = df
                print(f"Loaded {len(df)} records for record set {record_set_id}.")
            else:
                print(f"No records loaded for record set {record_set_id}.")
        except Exception as e:
            print(f"Error loading records for {record_set_id}: {e}")
    # Display the columns of the first available DataFrame
    if dfs:
        first_rs_id = list(dfs)[0]
        print(f"\nColumns in DataFrame for {first_rs_id}: \n{dfs[first_rs_id].columns.tolist()}")
        dfs[first_rs_id].head()
else:
    print("No record sets available to extract data from.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# We'll continue using the first DataFrame (if available) for EDA
if dfs:
    record_set_id = list(dfs)[0]
    df = dfs[record_set_id]

    # Try to find a likely numeric field for analysis
    numeric_fields = df.select_dtypes(include='number').columns.tolist()
    if numeric_fields:
        numeric_field = numeric_fields[0]  # Just pick the first for demonstration
        print(f"Analyzing numeric field: {numeric_field}\n")
        threshold = df[numeric_field].mean()  # Use the mean as an example threshold
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        print(filtered_df[[numeric_field]].head())

        # Normalization
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Grouping by a categorical field (if present)
        category_fields = df.select_dtypes(include='object').columns.tolist()
        group_field = None
        for c in category_fields:
            if c != numeric_field:
                group_field = c
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"\nGrouped mean of {numeric_field} by {group_field}:")
            print(grouped_df.head())
        else:
            print("No suitable categorical field found for grouping.")
    else:
        print("No numeric fields found in the data for EDA.")
else:
    print("No dataframes available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dfs:
    record_set_id = list(dfs)[0]
    df = dfs[record_set_id]
    numeric_fields = df.select_dtypes(include='number').columns.tolist()
    if numeric_fields:
        numeric_field = numeric_fields[0]
        plt.figure(figsize=(8, 4))
        sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
        plt.title(f'Distribution of {numeric_field}')
        plt.xlabel(numeric_field)
        plt.ylabel('Frequency')
        plt.show()

        # Joint plot with another numeric if available
        if len(numeric_fields) > 1:
            plt.figure(figsize=(6, 6))
            sns.scatterplot(x=df[numeric_fields[0]], y=df[numeric_fields[1]])
            plt.title(f"Scatter plot: {numeric_fields[0]} vs {numeric_fields[1]}")
            plt.xlabel(numeric_fields[0])
            plt.ylabel(numeric_fields[1])
            plt.show()
    else:
        print("No numeric fields found for plotting.")
else:
    print("No dataframes available for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

**In this notebook, we loaded the FAIR^2 dataset describing mass spectrometry analysis of extracellular vesicles from stimulated human mast cells using standardized Croissant metadata. We explored the dataset structure, previewed record sets, extracted a tabular view of the data, performed simple EDA and filtering, and visualized key data distributions.**

_This approach can be extended for deeper biological or statistical analysis of proteomics data. For full interpretation, please consult the dataset documentation and any accompanying publications._